# Explain bounds before comparing solvers
# Bounds의 의미를 확인하고 정식화 비교하기

Embed the same frozen 2→12→12→3 reactor network using manual constraints and OMLT. Inputs are T∈[310,350] K and tau∈[2,10] min; feed is fixed at 1.2 mol/L. Require B≥0.38, C≤0.25, and nonnegative outputs. Minimize tau+0.02(T−310). Investigate how valid bounds change the LP relaxation while preserving the network.

같은 고정 2→12→12→3 반응기 신경망을 수동 제약과 OMLT로 표현합니다. 입력은 T∈[310,350] K, tau∈[2,10]분이고 feed=1.2 mol/L입니다. B≥0.38, C≤0.25와 비음수 출력을 요구하며 tau+0.02(T−310)을 최소화합니다. 유효한 bounds가 신경망을 유지하면서 LP 완화를 어떻게 바꾸는지 확인합니다.

## Setup / 실행 준비

Extract the complete lab ZIP and run this notebook in that folder, with the scripts and supplied models beside it. Install requirements.txt into this notebook’s Python environment. The cells save new results under guided-results/.

실습 ZIP 전체를 풀고 스크립트·제공 모델과 같은 폴더에서 notebook을 실행합니다. Notebook의 Python 환경에 requirements.txt를 설치합니다. 새 결과는 guided-results/에 저장됩니다.

```text
python -m pip install -r requirements.txt
```

## 1. Propagate signs through one affine bound / 1. Affine bound에서 부호 따라가기

For a=2x1−3x2+0.5 and x1,x2∈[0,1], the smallest value uses x1=0,x2=1; the largest uses x1=1,x2=0. Positive weights multiply lower bounds for a lower estimate; negative weights multiply upper bounds. This is the calculation performed layer by layer.

a=2x1−3x2+0.5, x1,x2∈[0,1]이면 최솟값은 x1=0,x2=1에서, 최댓값은 x1=1,x2=0에서 나옵니다. 하한을 구할 때 양의 가중치에는 입력 하한을, 음의 가중치에는 입력 상한을 곱합니다. 이 계산을 층마다 반복합니다.

In [1]:
from pathlib import Path
import json
import numpy as np
import week06_lab as lab
lo, hi = np.zeros(2), np.ones(2)
w = np.array([2., -3.]); b = .5
lower = lo@np.maximum(w,0)+hi@np.minimum(w,0)+b
upper = hi@np.maximum(w,0)+lo@np.minimum(w,0)+b
print("affine bounds:", lower, upper, "ReLU bounds:", max(0,lower), max(0,upper))
np.testing.assert_allclose([lower,upper], [-2.5,2.5])
net = json.loads(Path("frozen_cstr_network.json").read_text())
bounds = lab.interval_bounds(net)
print("first three neuron bounds:", list(zip(bounds[0][0][:3],bounds[0][1][:3])))

affine bounds: -2.5 2.5 ReLU bounds: 0 2.5
first three neuron bounds: [(np.float64(-0.02939148574185513), np.float64(3.3931567318453126)), (np.float64(0.13643988524940132), np.float64(2.496094644317769)), (np.float64(-0.024893393054529374), np.float64(0.9775781307274805))]


After ReLU the interval is [0,2.5]. A preactivation upper bound ≤0 fixes a neuron off; a lower bound ≥0 makes it linear. Only intervals crossing zero need binaries. Broad independent intervals may miss correlations between neurons.

ReLU를 거치면 [0,2.5]입니다. Preactivation 상한이 0 이하면 꺼진 뉴런이고 하한이 0 이상이면 선형입니다. 0을 가로지르는 뉴런에만 이진 변수가 필요합니다. 독립 구간은 뉴런 사이의 상관관계를 놓칠 수 있습니다.

## 2. Explain the two-neuron relaxation gap / 2. 두 뉴런의 완화 차이 설명하기

The toy model has x1+x2=1, so a0=x1+x2−1=0 exactly. The other activation is max(0,x1−x2)=max(0,2x1−1). Requiring total activation ≥0.4 gives the exact solution x1≥0.7. A loose LP can inflate activations; fixing the first activation to zero removes one source of inflation.

작은 예제에서 x1+x2=1이므로 a0=x1+x2−1=0입니다. 다른 활성값은 max(0,x1−x2)=max(0,2x1−1)입니다. 활성값 합을 0.4 이상 요구하면 정확한 모델에서는 x1≥0.7입니다. 느슨한 LP는 활성값을 부풀릴 수 있고 첫 활성값을 0으로 고정하면 그 여지가 줄어듭니다.

In [2]:
toy = {}
for tight in [False,True]:
    for relax in [True,False]:
        result = lab.toy_case(tight,relax)
        toy[tight,relax] = result["objective"]
        print("tight, LP, objective:", tight,relax,result["objective"])
np.testing.assert_allclose([toy[False,True],toy[True,True],toy[True,False]], [0,.4,.7], atol=1e-7)

tight, LP, objective: False True 0.0
tight, LP, objective: False False 0.7
tight, LP, objective: True True 0.4
tight, LP, objective: True False 0.7


LP lower bounds improve from 0 to 0.4; the MILP optimum remains 0.7. The tightened LP is still a relaxation, not an exact forward graph. A stronger lower bound can reduce search, but preprocessing also takes time.

LP 하한은 0에서 0.4로 좋아지고 MILP 최적값은 0.7로 유지됩니다. 강화한 LP도 여전히 완화식입니다. 하한이 좋아지면 탐색을 줄일 수 있지만 전처리 시간도 듭니다.

## 3. Follow physical scaling and layer outputs / 3. 물리 입력 scaling과 층별 출력 확인하기

At [330 K,6 min], subtract [310,2] and divide by [40,8] to obtain [0.5,0.5]. Each saved weight matrix maps a row of activations to the next layer. Apply ReLU only where the layer metadata says relu; the output layer stays linear.

[330 K,6분]에서 [310,2]를 빼고 [40,8]로 나누면 [0.5,0.5]입니다. 저장된 가중치 행렬은 활성값 한 행을 다음 층으로 옮깁니다. Metadata가 relu인 층에만 ReLU를 적용하고 출력층은 선형으로 둡니다.

In [3]:
physical = np.array([330.,6.])
state = (physical-net["offset"])/net["scale"]
print("scaled input:", state)
for index, layer in enumerate(net["layers"]):
    pre = state@np.asarray(layer["weights"])+np.asarray(layer["bias"])
    state = np.maximum(pre,0) if layer["activation"] == "relu" else pre
    print("layer",index,"first preactivations",pre[:3],"first outputs",state[:3])
np.testing.assert_allclose(state,lab.forward(physical,net))
fixed = lab.solve_report(lab.make_omlt(net,fixed=[.5,.5],specifications=False))
assert fixed["has_incumbent"]
np.testing.assert_allclose(fixed["outputs"],state,atol=1e-6)

scaled input: [0.5 0.5]
layer 0 first preactivations [1.68188262 1.31626726 0.47634237] first outputs [1.68188262 1.31626726 0.47634237]
layer 1 first preactivations [ 0.76040616 -1.0993442  -0.14767557] first outputs [0.76040616 0.         0.        ]
layer 2 first preactivations [0.55071355 0.40560071 0.24312574] first outputs [0.55071355 0.40560071 0.24312574]


The direct forward output and fixed-input OMLT output should agree within tolerance. This checks scaling, matrix orientation, and activation encoding without mixing in product feasibility. The fixed argument uses scaled coordinates [0.5,0.5].

직접 forward와 입력 고정 OMLT 출력이 허용오차 안에서 같아야 합니다. 제품 규격 가능성과 분리해 scaling·행렬 방향·활성식 구현을 확인합니다. fixed 인수에는 scaling된 [0.5,0.5]를 넣습니다.

## 4. Compare complete reports, including failures / 4. 실패 상태까지 포함해 보고서 비교하기

The full run compares loose, interval, LP-tightened, and OMLT formulations using identical coefficients. LP-OBBT solves auxiliary LPs to bound each preactivation over the declared input domain. Record bound computation, model construction, and solve time separately.

전체 실행은 같은 계수로 느슨한 bounds, 구간 전파, LP 강화, OMLT를 비교합니다. LP-OBBT는 입력 영역에서 보조 LP를 풀어 각 preactivation의 범위를 구합니다. Bound 계산·모델 구성·solver 시간을 나눠 기록합니다.

In [4]:
import subprocess, sys
subprocess.run([sys.executable,"week06_lab.py","--output-dir","guided-results"],check=True)
report = json.loads(Path("guided-results/results.json").read_text())
for row in report["comparisons"]:
    print(row["formulation"],"LP bound",row.get("lp_objective"),"incumbent",row["objective"],
          "binaries",row["binary_variables"],"total seconds",row["total_seconds"])
print("infeasible:",report["infeasible_case"])
print("zero time limit:",report["zero_time_limit_case"])
assert report["max_forward_error"] < 1e-5
assert not report["infeasible_case"]["has_incumbent"]

loose LP bound 2.0 incumbent 3.1656262770140167 binaries 24 total seconds 0.34364080000523245
interval LP bound 2.0 incumbent 3.1656262770140398 binaries 22 total seconds 0.16438629999902332
LP-OBBT LP bound 2.5214932048437495 incumbent 3.1656262770141583 binaries 19 total seconds 0.25199019999126904
OMLT LP bound None incumbent 3.165626277014039 binaries 24 total seconds 0.10685890000604559
infeasible: {'termination': 'infeasible', 'has_incumbent': False, 'objective': None, 'bound': None, 'solve_seconds': 0.01916940000228351, 'binary_variables': 22, 'variables': 75, 'constraints': 100, 'absolute_gap': None, 'relative_gap': None}
zero time limit: {'termination': 'maxTimeLimit', 'has_incumbent': False, 'objective': None, 'bound': None, 'solve_seconds': 0.011423200005083345, 'binary_variables': 22, 'variables': 75, 'constraints': 100, 'absolute_gap': None, 'relative_gap': None}


The feasible optimal objectives should agree while relaxation bounds and binary counts can differ. Compare total time, including OBBT, and avoid drawing a speed ranking from one run. An absent incumbent means no selected inputs or gap can be reported. Submit a decoded bound, toy calculation, layer trace, equivalence error, solver status, and reference outputs at the selected point.

가능한 최적값은 일치해야 하지만 완화 하한과 이진 변수 수는 다를 수 있습니다. OBBT를 포함한 총 시간을 비교하고 한 번의 실행으로 속도 순위를 정하지 않습니다. Incumbent가 없으면 선택 입력이나 gap을 보고할 수 없습니다. Bound 계산, 작은 예제, 층별 추적, 일치 오차, 종료 상태, 선택점의 기준 출력을 제출하세요.

## Report / 보고서

Explain a hand prediction, the corresponding computed intermediate values, and the final interpretation. Change one setting and compare your predicted effect with the observed result. Keep training, validation/calibration, and held-out evaluation roles separate.

손으로 예측한 값, 대응하는 중간 계산값, 최종 해석을 연결해 설명합니다. 설정 하나를 바꾸고 예상한 효과와 실제 결과를 비교합니다. Train·validation/calibration·독립 평가 데이터의 역할을 구분합니다.